In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

train_path = Path("../data/Train/Train.csv")
df = pd.read_csv(train_path)
num_entries = len(df)
print(f"Loaded {num_entries} rows from {train_path}")
df.head()

# Preprocessed data summary

In [ ]:
from pathlib import Path
import pandas as pd

train_path = Path("../data/Processed/Train/Train.csv")
df = pd.read_csv(train_path)
num_entries = len(df)
print(f"Loaded {num_entries} rows from {train_path}")
df.head()

In [ ]:
test_path = Path("../data/Test/Test.csv")
df = pd.read_csv(test_path)
num_entries = len(df)
print(f"Loaded {num_entries} rows from {train_path}")
df.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(df["LeafArea"], df["DryWeightShoot"], alpha=0.7)
plt.title("Dry Weight vs Leaf Area")
plt.xlabel("Leaf Area")
plt.ylabel("Dry Weight Shoot")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

depth_dir = Path("../data/Processed/Train/Depth")
try:
    sample_id = int(df["image_id"].iloc[0])
except Exception:
    sample_id = 15

depth_path = depth_dir / f"{sample_id}.npy"
depth_map = np.load(depth_path)

# check if normalised
print(f"max: {depth_map.max()}  min: {depth_map.min()}  mean: {depth_map.mean():.4f}  std: {depth_map.std():.4f}")


plt.figure(figsize=(6, 6))
plt.imshow(depth_map, cmap="viridis")
plt.title(f"Depth map for image_id {sample_id}")
plt.axis("off")
plt.colorbar(label="Normalized depth")
plt.show()

# Preprocessing dimensions

In [ ]:
import cv2
from pathlib import Path
import matplotlib.pyplot as plt
from preprocessing import center_square_crop   # ensure yours has scale support

# Different crop scales to visualise
CROP_SCALES = [1.0, 0.9, 0.7, 0.5]   # 1.0 = full square, smaller = tighter crop

SAMPLE_ID = 375
rgb_dir = Path("../data/Train/RGBImages")
rgb_path = rgb_dir / f"RGB_{SAMPLE_ID}.png"

# Load image
rgb_bgr = cv2.imread(str(rgb_path))
if rgb_bgr is None:
    raise FileNotFoundError(f"Could not read {rgb_path}")

rgb = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2RGB)

# Prepare figure
fig, axes = plt.subplots(1, len(CROP_SCALES) + 1, figsize=(4 * (len(CROP_SCALES) + 1), 4))

# Original first
axes[0].imshow(rgb)
axes[0].set_title("Original (no crop)")
axes[0].axis("off")

# Visualise each crop scale
for ax, scale in zip(axes[1:], CROP_SCALES):
    cropped = center_square_crop(rgb, scale=scale)
    ax.imshow(cropped)
    ax.set_title(f"Crop scale {scale}")
    ax.axis("off")

plt.tight_layout()
plt.show()


# Displaying cropped images

In [ ]:

def display_processed_images(csv_path, processed_root, mode='Train', max_images=None):
    """
    Display RGB and depth pairs from the processed dataset to visually inspect preprocessing.
    """
    df = pd.read_csv(csv_path)
    rgb_dir = Path(processed_root) / mode / 'RGB'
    depth_dir = Path(processed_root) / mode / 'Depth'

    for idx, row in df.iterrows():
        if max_images is not None and idx >= max_images:
            break

        image_id = str(int(row['image_id']))
        rgb_path = rgb_dir / f"{image_id}.png"
        depth_path = depth_dir / f"{image_id}.npy"

        fig, axes = plt.subplots(1, 2, figsize=(8, 4))

        if rgb_path.exists():
            rgb_img = cv2.cvtColor(cv2.imread(str(rgb_path)), cv2.COLOR_BGR2RGB)
            axes[0].imshow(rgb_img)
            axes[0].set_title(f"RGB {image_id}")
        else:
            axes[0].text(0.5, 0.5, f"Missing {rgb_path.name}", ha='center', va='center')
            axes[0].set_title("RGB missing")
        axes[0].axis("off")

        if depth_path.exists():
            depth_map = np.load(depth_path)
            axes[1].imshow(depth_map, cmap='viridis')
            axes[1].set_title(f"Depth {image_id}")
        else:
            axes[1].text(0.5, 0.5, f"Missing {depth_path.name}", ha='center', va='center')
            axes[1].set_title("Depth missing")
        axes[1].axis("off")

        plt.tight_layout()
        plt.show()
        plt.close(fig)
    
display_processed_images('../data/Processed/Train/Train.csv', '../data/processed', mode='Train', max_images = 50)